# aw_01_G — Gates G1/G2/G3: runtime hardening, verifier freeze, eval-suite freeze

**Protocol**: §10 (gates), §7 (frozen suites). Run ONCE per protocol version.

- **G1**: end-to-end smoke (config → data → train step → eval step) on a tiny budget,
  PLUS two amendments added after Phase-2 incidents:
  - **G1-reward-transport** (2026-08-15 B6 incident): heterogeneous-family GRPO
    scenario-transport audit (x17). Arrow struct unification only manifests when
    families with different key sets are mixed — a single-family tiny smoke cannot
    catch it, so the audit runs on the real frozen prompts file.
  - **G1-env-attestation** (2026-08-16 ABI incident): vLLM compatibility probe (x18,
    dry-run — does NOT mutate the runtime). Finds the newest vLLM whose resolver
    plan leaves the image-owned ABI layer (torch/torchvision/torchaudio/triton/
    nvidia-*) untouched. The resulting exact pin is committed to
    `requirements/vllm.lock.txt`; experiment notebooks only CONSUME that pin
    (aw_09_b6 header installs vLLM only if a `vllm==` line exists).
- **G2**: verifier freeze — the verifier regression suite (expected-status fixtures:
  pass/fail/malformed/illegal/timeout/indeterminate + reward-bridge semantics) must
  pass 100%; tag the verifier version in the protocol log.
- **G3**: build and FREEZE the five eval suites (300 episodes each) and the freeze
  manifest. Commit `data/eval_suites/freeze_manifest.json`; training loaders must
  treat the eval family ids as `forbidden_family_ids` (leakage gate).

**Outputs**: freeze_manifest.json (committed), smoke run artifacts,
`runs/x17_gate_g1.json`, `runs/x18_vllm_probe.json`, vLLM pin in
`requirements/vllm.lock.txt`.

In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


Cloning into 'axiom-world'...
remote: Enumerating objects: 742, done.
remote: Counting objects: 100% (291/291), done.
remote: Compressing objects: 100% (192/192), done.
remote: Total 742 (delta 175), reused 184 (delta 83), pack-reused 451 (from 1)
Receiving objects: 100% (742/742), 325.29 KiB | 7.93 MiB/s, done.
Resolving deltas: 100% (407/407), done.
/content/axiom-world
Obtaining file:///content/axiom-world
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 66.2 MB/s eta 0:00:00
  Building editable for axiom-world (pyproject.toml) ... don

In [ ]:
# @title 01a_g1_smoke
!python scripts/smoke_gate_g1.py


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
{
  "python": "3.12.13",
  "cuda_available": true,
  "gpu": {
    "name": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
    "capability": [
      12,
      0
    ],
    "vram_gb": 94.97,
    "sm_count": 188
  }
}
TRL version: 1.9.0
config.json: 100% 729/729 [00:00<00:00, 7.88MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 7.32MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 15.3MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 72.8MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 316MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetenso

In [ ]:
# @title 01a_g1_reward_transport_smoke — heterogeneous-family GRPO reward path (x17)
# G1 amendment (2026-08-15 B6 incident): proves the scenario transport used by
# GRPO training is Arrow-safe on the REAL frozen prompts file:
#  - legacy dict column: contamination evidence (rows mutated / None-injected)
#  - fixed scenario_json string column: 0 mismatches, 0 validation failures
#  - reward parity: oracle completions score identically with and without Arrow
# Requires the frozen prompts artifact (fetch per protocol v1.3 sha-pinning first
# if not present):
#   python scripts/fetch_dataset.py --repo m97j/axiom-playworld \
#       --file playworld_prompts.jsonl --sha256 <frozen sha> --out data/train/
!python scripts/fetch_dataset.py \
  --repo m97j/aw-playworld --path train/v1/playworld_prompts.jsonl \
  --output data/train/playworld_prompts.jsonl --force \
  --expected-sha256 2e7d02603c47784328ee82bba8abb6ed8b9e32175567b72b7a68969a0ae361e6

!python scripts/x17_grpo_scenario_audit.py \
  --prompts data/train/playworld_prompts.jsonl \
  --out runs/x17_gate_g1.json
# verdict must be PASS (exit 0). Non-zero exit = gate FAIL — do not proceed to training.

playworld_prompts.jsonl: 100% 4.60M/4.60M [00:00<00:00, 25.4MB/s]
fetched dataset: hf://m97j/aw-playworld/train/v1/playworld_prompts.jsonl
revision: main
materialized: data/train/playworld_prompts.jsonl
dataset sha256: 2e7d02603c47784328ee82bba8abb6ed8b9e32175567b72b7a68969a0ae361e6
DATASET_PATH=data/train/playworld_prompts.jsonl
DATASET_SHA256=2e7d02603c47784328ee82bba8abb6ed8b9e32175567b72b7a68969a0ae361e6
{
  "prompts": "data/train/playworld_prompts.jsonl",
  "n_rows": 2000,
  "legacy_dict_column": {
    "rows_mutated_by_arrow": 2000,
    "validation_failures": 2000
  },
  "fixed_json_string_column": {
    "roundtrip_mismatches": 0,
    "validation_failures": 0
  },
  "reward_parity": {
    "n_scored": 64,
    "n_equal": 64,
    "direct_none": 0,
    "fixed_none": 0,
    "fixed_mean_reward": 1.0
  },
  "verdict": "PASS"
}


In [ ]:
# @title 01a_g1_env_attestation — image-owned ABI baseline + vLLM compat probe (x18)
# G1 amendment (2026-08-16 ABI incident): a floating vllm range replaced
# torch 2.11.0+cu128 with 2.13.0+cu13x while torchaudio stayed cu128 →
# Qwen3ForCausalLM import failure. This probe is DRY-RUN ONLY (no runtime
# mutation): it snapshots the ABI baseline and finds the newest vLLM whose pip
# resolver plan touches NO image-owned package.
!python scripts/x18_vllm_compat_probe.py --out runs/x18_vllm_probe.json --max-candidates 50

# If a CLEAN candidate is reported, OPTIONALLY smoke-test + benchmark it in THIS
# gate runtime (mutates the env — this runtime is disposable, training is not):
#   !python scripts/x18_vllm_compat_probe.py --install --model Qwen/Qwen3-8B \
#       --out runs/x18_vllm_probe_install.json
# Then write the reported lock_line (vllm==X) into requirements/vllm.lock.txt
# and commit it together with runs/x18_vllm_probe*.json.
# If NO candidate is CLEAN: leave the lock unpinned; aw_09_b6 header will skip
# the vLLM install and training must run with
#   --override training.extra.use_vllm=false   (HF-generate fallback).

vllm==0.27.1: ABI-TOUCH ['nvidia-cutlass-dsl==4.6.0', 'torch==2.13.0', 'torchvision==0.28.0', 'nvidia-cuda-nvrtc==13.0.88', 'nvidia-cuda-runtime==13.0.96', 'nvidia-cudnn-cu13==9.20.0.48', 'nvidia-cusparselt-cu13==0.8.1', 'nvidia-cutlass-dsl-libs-base==4.6.0', 'nvidia-cutlass-dsl-libs-cu12==4.6.0', 'nvidia-cutlass-dsl-libs-cu13==4.6.0', 'nvidia-nccl-cu13==2.29.7', 'nvidia-nvshmem-cu13==3.4.5', 'triton==3.7.1', 'nvidia-cublas==13.1.1.3', 'nvidia-cuda-cupti==13.0.85', 'nvidia-cufft==12.0.0.61', 'nvidia-cufile==1.15.1.6', 'nvidia-curand==10.4.0.35', 'nvidia-cusolver==12.0.4.66', 'nvidia-cusparse==12.6.3.3', 'nvidia-cutlass-dsl-libs-core==4.6.0', 'nvidia-nvtx==13.0.85', 'nvidia-cudnn-frontend==1.27.0', 'torchcodec==0.16.0', 'nvidia-cuda-cccl==13.3.3.4.1', 'nvidia-cuda-nvcc==13.3.73', 'torch_c_dlpack_ext==0.1.5', 'nvidia-cuda-nvdisasm==13.3.73', 'nvidia-nvjitlink==13.3.33', 'nvidia-cuda-crt==13.3.73', 'nvidia-nvvm==13.3.73']
vllm==0.27.0: ABI-TOUCH ['nvidia-cutlass-dsl==4.6.0', 'torch==2.13.

In [ ]:
# @title 01a_g1_vllm_server_mode_test
!bash scripts/launch_trl_vllm_server.sh Qwen/Qwen3-8B 8000

[launch] creating isolated venv at /content/vllm-env
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 142.4 MB/s eta 0:00:00
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: /content/vllm-env
Activate with: source /content/vllm-env/bin/activate
Using Python 3.12.13 environment at: /content/vllm-env
Resolved 203 packages in 952ms
Prepared 203 packages in 24.13s
Installed 203 packages in 373ms
 + accelerate==1.14.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==0.122.0
 + anyio==4.14.2
 + apache-tvm-ffi==0.1.9
 + astor==0.8.1
 + attrs==26.1.0
 + blake3==1.0.9
 + cachetools==7.1.7
 + cbor2==6.1.4
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.1
 + click==8.4.2
 + cloudpickle==3.1.2
 + compressed-tensors==0.17.0
 + cryptography==50.0.0
 + cuda-bindings==13.3.1
 + cuda-core==1.0.1
 + cuda-pathfinder==1.6.0
 + cuda-python==13.3.1
 + cuda-tile==1.3

In [ ]:
# @title 01b_g2_verifier_freeze — verifier regression suite (100% required)
# G2 pass condition (protocol §10): every expected-status fixture agrees —
# pass/fail/malformed/illegal/timeout/indeterminate — across the PlayWorld
# verifiers, the general verifier, and the reward-bridge status→reward map.
# Any failure = gate FAIL: fix, re-run, and only then tag the verifier version.
!python -m pytest tests/unit/test_verifiers.py \
                  tests/unit/test_general_verifier.py \
                  tests/unit/test_reward_bridge.py -q
# After passing: Record verifier version tag in protocol log
!git rev-parse --short HEAD | xargs -I{} echo "G2 verifier freeze tag: verifier-{}"

...................                                                      [100%]
19 passed in 0.05s
G2 verifier freeze tag: verifier-a3cf2a7


In [ ]:
# @title 01c_g3_eval_freeze
!python scripts/build_eval_suites.py --episodes-per-suite 300
# Commit data/eval_suites/freeze_manifest.json to the repo after this cell.


eval_id: 300 episodes -> data/eval_suites/eval_id.jsonl (sha256:aceeea727d2b9eaed...)
eval_template_ood: 300 episodes -> data/eval_suites/eval_template_ood.jsonl (sha256:13580a6cbf7a4e566...)
eval_comp_ood: 300 episodes -> data/eval_suites/eval_comp_ood.jsonl (sha256:444191a244dcd77d1...)
eval_rule_ood: 300 episodes -> data/eval_suites/eval_rule_ood.jsonl (sha256:d73745108f5e7e207...)
eval_adversarial: 300 episodes -> data/eval_suites/eval_adversarial.jsonl (sha256:c73dd155acd069292...)

G3 freeze manifest -> data/eval_suites/freeze_manifest.json
Commit this manifest; training loaders must pass eval_family_ids as forbidden_family_ids (leakage gate).


## Gate checklist
- [x] G1 smoke passed (no exceptions, artifacts written)
- [x] G1 reward-transport audit (x17) verdict PASS on the frozen prompts file — `runs/x17_gate_g1.json` archived
- [x] G1 env attestation (x18) run — baseline recorded; exact `vllm==` pin committed to `requirements/vllm.lock.txt` (or explicitly left unpinned → HF-generate fallback documented)
- [x] G2 verifier regression suite 100% pass — verifier version tag recorded in the protocol log
- [x] G3 freeze manifest committed — fingerprint recorded in the protocol log